# Embeddings & Semantic Search

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/building-with-llms/03-embeddings-and-semantic-search

A from-scratch, runnable implementation of the concepts in the lesson — pure NumPy, no API keys required.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import re

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## Toy embeddings

Real embeddings come from a trained transformer. To stay dependency-free we use a simple bag-of-words vector over a shared vocabulary — enough to demonstrate cosine ranking. The *mechanics* (vector + cosine + ranking) are identical to a production retriever.

In [ ]:
CORPUS = [
    'Refunds are available within 30 days of purchase',
    'How to request a refund for your order',
    'International shipping options and delivery times',
    'Track your package after it ships',
    'Reset your password from the account settings page',
]

def tokenize(s):
    return re.findall(r'[a-z]+', s.lower())

vocab = sorted({w for d in CORPUS for w in tokenize(d)})
index = {w: i for i, w in enumerate(vocab)}

def embed(text):
    v = np.zeros(len(vocab))
    for w in tokenize(text):
        if w in index:
            v[index[w]] += 1.0
    return v

doc_vecs = np.array([embed(d) for d in CORPUS])
print('vocab size:', len(vocab), '| doc matrix:', doc_vecs.shape)

## Cosine similarity and ranking

In [ ]:
def cosine(a, b):
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na == 0 or nb == 0:
        return 0.0
    return float(a @ b / (na * nb))

def search(query, k=3):
    q = embed(query)
    sims = [cosine(q, d) for d in doc_vecs]
    order = np.argsort(sims)[::-1][:k]
    return [(CORPUS[i], round(sims[i], 3)) for i in order]

for hit in search('how do I get my money back', k=3):
    print(hit)

Note the top hits are about *refunds* even though the query never says 'refund' — meaning, not keywords.

## ✏️ Your turn

Implement `top_k_retrieve(query_vec, doc_vecs, k)` returning the **indices** of the k nearest docs by cosine.

In [ ]:
def top_k_retrieve(q, docs, k):
    # TODO(you): score each doc by cosine(q, doc), return indices of the top k (descending).
    return []

idx = top_k_retrieve(embed('package tracking'), doc_vecs, 2)
assert 3 in idx  # 'Track your package after it ships'
assert len(idx) == 2
print('passed ✓')

<details><summary>Solution</summary>

```python
def top_k_retrieve(q, docs, k):
    sims = [cosine(q, d) for d in docs]
    return list(np.argsort(sims)[::-1][:k])
```

</details>